Phase 2 Begins

Phase 1 built the data pipelines. Phase 2 addes AI on top

Today's pipeline:

Messy Invoice Text

Prompt Engineering                Design the precise instructions
       
GroQ API                          Call LLM (Llama 3.1)

Json Parsing                      Extract structured data

Pandas Dataframe                  Clean, analysable table

Analysis                          Business insights

Tools : Groq API. Python groq library, json, Pandas

API KEY = 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

In [7]:
#CELL 1
!pip install groq --quiet

import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

print('Libraries imported successfully')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.7 MB/s eta 0:00:00
Libraries imported successfully


API = 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

In [8]:
from groq import Groq

API_KEY = "gsk_llhJUnpFy8VPCZT1PDccWGdyb3FYoHghb0fllSGrtrNL4ZA9Hg"

client = Groq(api_key=API_KEY)

MODEL = "llama-3.1-8b-instant"

print(f'Groq Client configured with model: {MODEL}')
print('Make sure API_KEY is replaced with you actual key!')

Groq Client configured with model: llama-3.1-8b-instant
Make sure API_KEY is replaced with you actual key!


In [9]:
#CELL 3

def ask_llm(user_message, system_message='You are a helpful assistant.',temperature=0.7, max_tokens=500):
    """
    Send a message to the LLM and return the response text.

    Parameters:
    ----------------------
    user_message :
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            #system messages = instructions to the modek about how to behave
            #Like telling a new employee: 'You work at teh data company'
            {"role": "user", "content": user_message},
        ],
        temperature=temperature,
        #temperature controls randomness
        #0.0 = always picks the most likely next token (consisten)
        max_tokens=max_tokens
        #maximum number od tokens in the response
        #500 tokens = 375 words
    )
    return response.choices[0].message.content
    #response.choices = list of possible completions (we requested 1)
    #[0]              = first completion
    #.message.content = the actual text the LLM generated

#Test the connection
test_response=ask_llm(
    "What is ETL in data engineering? Answer in exactly 2 sentences."
)
test_response2 = ask_llm(
    "Is GenAI abd Data Engineering a good career in 2026?"
)
print('=== LLM Response')
print(test_response)
print(test_response2)

=== LLM Response
ETL (Extract, Transform, Load) is a data engineering process used to retrieve data from various sources, transform it into a standardized format, and load it into a target system, such as a data warehouse or database, for analysis and storage. ETL helps to ensure data consistency, quality, and accuracy by handling data integration, cleaning, and processing in a structured and scalable manner.
Based on current trends and the job market in 2026, GenAI (General Artificial Intelligence) and Data Engineering are excellent career choices. Here's why:

**GenAI:**

1. **Rapid growth**: As AI continues to transform industries, the demand for professionals who can develop and work with GenAI systems is skyrocketing. According to a report by ResearchAndMarkets, the global AI market is expected to reach $190 billion by 2025.
2. **High salaries**: GenAI professionals, such as AI engineers and researchers, can earn salaries ranging from $150,000 to over $250,000 per year, depending 

In [10]:
response_etl = ask_llm(
    "In a 3 bullet points, explain how the Medallion Architecture "
    "(Bronze, Silver, Gold layer) related to ETL pipelines.",
    system_message="you are a senior data engineering instructor."
                   "Be concise and practical."
)
print('Medallion + ETL connection:')
print(response_etl)
print()
print("--- Token explanation ---")
print('Each word is roughly 1-2 tokens.')
print('The model above used approximately', len(response_etl.split())*1.3, 'tokens.')
print('Llama-3.1-8b context window: 8192 tokens (~6000 words per conversation)')

Medallion + ETL connection:
Here are three key points about the Medallion Architecture (Bronze, Silver, Gold layer) in relation to ETL pipelines:

* **Bronze Layer (Raw Data Ingestion)**: This layer is responsible for ingesting raw data from various sources, such as databases, APIs, or files. It's the entry point of the ETL pipeline and focuses on data collection, processing, and persistence. Data is typically stored in a raw, untransformed format in this layer.
* **Silver Layer (Data Processing and Enrichment)**: In this layer, raw data is transformed, processed, and enriched to make it more useful for analysis. This might involve data cleansing, aggregation, or joining with other datasets. The Silver layer provides a more refined and processed version of the data, which is then stored in a separate environment.
* **Gold Layer (Curated Data for Analysis)**: The Gold layer represents the final, polished version of the data, which is optimized for analysis and decision-making. This laye

In [11]:
zero_shot_response = ask_llm(
    "Extract the city name from this address:"
    "456 Bridge Road, Bangalore 560025, Karnataka, India"
)
print('Zero-Shot Result:')
print(zero_shot_response)

ambiguous_response =ask_llm("Clean this data: ramesh kumar,45000,mumbai")
print('Ambiguous Zero-Shot Result:')
print(ambiguous_response)
print()
print('Problem: output format is unpredictable and not machine-parseable!')

Zero-Shot Result:
The city name extracted from the given address is: Bangalore
Ambiguous Zero-Shot Result:
To clean the data, I would separate it into individual fields and identify potential issues. Here's the cleaned data:

1. Name: Ramesh Kumar
   - Issue: No middle name or last name separation, but it's possible to consider it as a single name field.

2. Salary: 45000
   - Issue: No currency symbol or unit mentioned, assuming it's in the local currency.

3. City: Mumbai
   - Issue: No state or region mentioned, assuming it's the main city.

Here's the data formatted for better readability:

Name: Ramesh Kumar
Salary: 45000
City: Mumbai

If you want to format the data in a more structured way with different fields for first name, last name, etc., I can do that as well. 

Here is an example:

Name: 
- First Name: Ramesh
- Middle Name: 
- Last Name: Kumar

Salary: 45000

City: Mumbai

Problem: output format is unpredictable and not machine-parseable!


In [12]:
same_question = "Review this python code and identify any issues:\n" \
                "df['revenue'] = df['qty'] * df['price']\n" \
                "result = df.groupby('dept').sum()"

generic_response = ask_llm(same_question, temperature=0.2)
print('Without Role Prompting:')
print(generic_response[:300], '...')
print()


role_response = ask_llm(
    same_question,
    system_message="You are a senior data engineer with 10 years of production"
                   "experience. Review code critically for production readiness"
                   "data types issues, and potential failures at scale.",
    temperature=0.2
)

print('With Role Prompting (Senior Data ENgineer):')
print(role_response[:400], '...')
print()
print("Notice: role prompting produces more technical, actionable feedback")
#

Without Role Prompting:
The provided Python code appears to be a basic data manipulation task using the pandas library. However, there are a few potential issues that can be identified:

1. **Missing Error Handling**: The code does not include any error handling. If the 'qty' or 'price' columns do not exist in the DataFram ...

With Role Prompting (Senior Data ENgineer):
**Code Review**

The provided Python code appears to be a basic data manipulation task using the pandas library. However, there are several potential issues that could impact production readiness:

### 1. Data Type Issues

The code assumes that the 'qty' and 'price' columns are numeric data types. However, if these columns contain non-numeric values, the multiplication operation will fail. To miti ...

Notice: role prompting produces more technical, actionable feedback


In [13]:
prompt = "Give me one creative name for a data analytics startup."

print('=== Temperature Experiment ===')
for temp in [0.0, 0.5, 1.0]:
  response = ask_llm(prompt, temperature=temp)
  print(f'Temperature = {temp}: {response.strip()}')
  time.sleep(1)

print()
print('Observation:')
print('temperature=0.0 -> same or very similar answer every run (deterministic)')
print('temperature=0.5 -> some variation')
print('temperature=1.0 -> more creative/varied, sometimes surprising')
print()
print('Rule for data engineering tasks: use temperature=0.0 or 0.1')
print('You need CONSISTENT, PARSEABLE output - not creative variation')
#

=== Temperature Experiment ===
Temperature = 0.0: Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect disparate data points and provide valuable insights. This name conveys the idea of a startup that helps businesses navigate complex data landscapes and uncover hidden patterns and trends.
Temperature = 0.5: Here's a creative name for a data analytics startup:

"Apexion Insights"

"Apexion" combines "apex" (the highest or most superior point) and "ion" (a charged particle), suggesting a high-energy and forward-thinking approach to data analysis. "Insights" conveys the idea of gaining valuable and actionable knowledge from data, which is at the heart of what a data analytics startup does.
Temperature = 1.0: One creative name for a data analytics startup is "Nexixa." This name suggests a connection or nexus between data points and insights, which aligns with the core purpose of a data analytics c

In [14]:
invoice_text = " Invoice #2024-001 from TECHWORLD SOLUTIONS dated 15th January 2024. Amount 45,000 for Laptop"

weak_response = ask_llm(
    f'Clean this invoice data: {invoice_text}',
    temperature=0.3
)
print('Weak Prompt Output:')
print(weak_response)
print()


try:
  json.loads(weak_response)
  print('PARSEABLE: Yes')
except:
  print('Parseable: No - cannot load into DataFrame')

print('\n' + '='*50 + '\n')

strong_system = """You are a data extraction specialist for an accounting pipeline.
Extract invoice data and return ONLY a valid JSON object.
DO NOT include any explanation, preamble, or markdown formatting.
Return ONLY the JSON, nothing else.

JSON schema (use null for missing values):
{"invoice_id": string, "vendor_name": string (Title Case),
"amount": number(no currency symbols),
"currency": string (default INr),
"invoice_date": string (YYYY-MM-DD),
"category": string (Electronics/Services/Accessories/Others)
}
"""

strong_response = ask_llm(
    f'Extract from: {invoice_text}',
    system_message=strong_system,
    temperature=0.0
)
print('Strong Prompt Output:')
print(strong_response)
print()
try:
  parsed = json.loads(strong_response.strip())
  print('PARSEABLE: Yes')
  print(f'Vendor: {parsed.get("vendor_name")}')
  print(f'Amount: {parsed.get("amount")}')
  print(f'Date: {parsed.get("invoice_date")}')
except json.JSONDecodeError:
  print('Parseable: No - cannot load into DataFrame')

Weak Prompt Output:
Here's the cleaned invoice data:

**Invoice Details:**

- **Invoice Number:** 2024-001
- **Date:** 15th January 2024
- **Company:** Techworld Solutions
- **Description:** Laptop
- **Amount:** $45,000

I reformatted the date to a more standard format (15 January 2024), and added a dollar sign to the amount for clarity.

Parseable: No - cannot load into DataFrame


Strong Prompt Output:
{"invoice_id": "2024-001", "vendor_name": "Techworld Solutions", "amount": 45000, "currency": "INR", "invoice_date": "2024-01-15", "category": "Electronics"}

PARSEABLE: Yes
Vendor: Techworld Solutions
Amount: 45000
Date: 2024-01-15


MINI PROJECT: Smart Data Cleaner

Goal: Convert 5 messy invoice strings into a clean, structured Pandas Dataframe using LLM

This is a complete GenAI-powered ETL pipeline.

Messy Text-> LLM -> JSON -> DataFrame ->Analysis

Q1. What is the difference between ML (day 5) and Generative AI(Day 6)

Q2. What does temperature=0.0 do in an LLM API call and when would you use it?

Q3. Write a few-shot prompt that extracts name and salary from text in JSON format.

Q4. What is LLM hallucination and how can prompt engineering reduce it?

Q5. Your LLM returns ```json\n{"name":"Ramesh"}\n and json.loads() crashes. Write the fix.

*Q6:*How does today's

### Short Answers to Your Questions:

#### Q1. Difference between ML and Generative AI
**ML** focuses on learning patterns from data to make predictions or decisions. **Generative AI** is a type of ML that creates new, original content (like text or images) based on learned patterns.

#### Q2. What does `temperature=0.0` do?
`temperature=0.0` makes the LLM's output highly deterministic and consistent (least creative). Use it for tasks requiring precise, repeatable results, such as data extraction and parsing.

#### Q3. Few-shot prompt for name and salary extraction in JSON

In [15]:
few_shot_prompt_short = '''
Convert employee text to JSON. Examples:
Input: John Doe, 60000
Output: {"name": "John Doe", "salary": 60000}

Input: Jane Smith, 75000
Output: {"name": "Jane Smith", "salary": 75000}

Input: ALICE JOHNSON, 82000
Output:
'''
print('Short Few-Shot Prompt:')
print(few_shot_prompt_short)
# Expected output for 'ALICE JOHNSON, 82000' would be: {"name": "Alice Johnson", "salary": 82000}

Short Few-Shot Prompt:

Convert employee text to JSON. Examples:
Input: John Doe, 60000
Output: {"name": "John Doe", "salary": 60000}

Input: Jane Smith, 75000
Output: {"name": "Jane Smith", "salary": 75000}

Input: ALICE JOHNSON, 82000
Output:



#### Q4. LLM hallucination and how prompt engineering reduces it
**LLM hallucination** is when the model generates false or nonsensical information. **Prompt engineering** reduces it by providing clear instructions, context, examples, and strict output constraints (e.g., "Answer only from provided text", "Return ONLY JSON").

#### Q5. Fix for `json.loads()` crashing on ```json\n{"name":"Ramesh"}\n

In [16]:
import json
import re

llm_output_bad = '```json\n{"name":"Ramesh"}\n```'

# Use regex to extract the pure JSON string
match = re.search(r'```json\n(.*)\n```', llm_output_bad, re.DOTALL)
if match:
    json_string_fixed = match.group(1).strip()
    try:
        parsed_data = json.loads(json_string_fixed)
        print(f"Fixed: Successfully parsed as: {parsed_data}")
    except json.JSONDecodeError as e:
        print(f"Error even after regex: {e}")
else:
    print("No JSON block found.")

Fixed: Successfully parsed as: {'name': 'Ramesh'}


#### Q6. Smart Cleaner vs. Manual ETL Cleaning
Today's **Smart Cleaner** (LLM-based) automatically extracts and structures data from *unstructured text* using natural language understanding, adapting flexibly to variations. **Manual ETL** relies on predefined rules for *structured/semi-structured data* and requires more effort to adapt to new formats.

In [17]:
# Make sure to run all preceding cells, especially the one defining `ask_llm`.
few_shot_prompt = """
Convert employee text to JSON. Here are examples:

Input: RAMESH KUMAR, 45000, mumbai
Output: {"name": "Ramesh Kumar", "salary": 45000, "city": "Mumbai"}

Input: priya nair, 52000, Delhi
Output: {"name": "Priya Nair", "salary": 52000, "city": "Delhi"}

Now convert this:
Input: ANANYA DAS, 38000, kolkata
Output:
"""
few_shot_response = ask_llm(few_shot_prompt, temperature=0.0)
print('Few-Shot Result:')
print(few_shot_response)
print()

try:
  # The type hint 'parsed: json.loads' is incorrect. It should be 'parsed = json.loads'
  parsed = json.loads(few_shot_response.strip())
  print('Sucessfully parsed as JSON!')
  print(f'Name: {parsed["name"]}, Salary: {parsed["salary"]}, City: {parsed["city"]}')

except json.JSONDecodeError:
  print('Parsing failed - model added extra text or invalid JSON.')
  print("Solution: add explicit instructions in the system prompt to return ONLY JSON, and check your prompt examples for correct JSON syntax.")


Few-Shot Result:
Here's a Python function that can convert the employee text to JSON:

```python
import json

def convert_to_json(employee_text):
    # Split the input string into individual values
    values = employee_text.split(', ')
    
    # Create a dictionary with the given keys
    employee_data = {
        "name": values[0].strip().title(),
        "salary": int(values[1]),
        "city": values[2].strip().title()
    }
    
    # Convert the dictionary to JSON
    json_data = json.dumps(employee_data)
    
    return json_data

# Test the function
employee_text = "ANANYA DAS, 38000, kolkata"
print(convert_to_json(employee_text))
```

When you run this function with the input "ANANYA DAS, 38000, kolkata", it will output:

```json
{"name": "Ananya Das", "salary": 38000, "city": "Kolkata"}
```

This function works by splitting the input string into individual values using the `split(', ')` method. It then creates a dictionary with the given keys and assigns the corresponding v